# glcuda T4 Wave 66 - compensated-MMA AV feasibility


In [ ]:
import base64
import hashlib
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import traceback
import urllib.request
import zipfile

BASE_REV = '3bce8dd7b8aaa2765855ab927c611b54981f9241'
REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
EMBEDDED = json.loads('[{"path":"glcuda/src/kernels/glcuda_sm75_wave64.ptx","sha256":"8279698f0e8c0df60ca84b73d9b608661bf6b2c36a56ef09be8f4c372157999c","base64":"LnZlcnNpb24gNy4wCi50YXJnZXQgc21fNzUKLmFkZHJlc3Nfc2l6ZSA2NAoKLy8gV2F2ZSA2NCBkaWFnbm9zdGljIG9ubHkuIFRoZXNlIGVudHJpZXMgYXJlIG5vdCBsb2FkZWQgYnkgS2VybmVsU2V0LgovLwovLyBJbnB1dCBwcm9iYWJpbGl0aWVzIGFyZSBwYWNrZWQgW2hlYWRzLCBudG9rLCBjYXBhY2l0eV0gd2l0aCBjYXVzYWwtdGFpbAovLyB6ZXJvcy4gVGhlIHJldGFpbmVkIGVudHJ5IHJlYWRzIHJvdy1tYWpvciBWIFtoZWFkcywgY2FwYWNpdHksIDY0XS4gVGhlCi8vIGNhbmRpZGF0ZSByZWFkcyBpdHMgbG9hZC10aW1lL3dyaXRlLXRpbWUgdHJhbnNwb3NlIFtoZWFkcywgNjQsIGNhcGFjaXR5XS4KLy8gQm90aCB3cml0ZSBwYWNrZWQgZjMyIFtoZWFkcywgbnRvaywgNjRdLgovLwovLyBMYXVuY2g6IGdyaWQgKGNlaWwobnRvay8xNiksIGhlYWRzKSwgYmxvY2sgMTI4LgovLyBDYW5kaWRhdGU6IG9uZSB3YXJwIG93bnMgdHdvIGFkamFjZW50IE44IGZyYWdtZW50cyBvZiB0aGUgMTZ4NjQgb3V0cHV0LgoKLnZpc2libGUgLmVudHJ5IGdsX3dhdmU2NF9hdl9zY2FsYXJfZjMyKAogICAgLnBhcmFtIC51NjQgcF9wcm9iLAogICAgLnBhcmFtIC51NjQgcF92LAogICAgLnBhcmFtIC51NjQgcF9vdXQsCiAgICAucGFyYW0gLnUzMiBwX250b2ssCiAgICAucGFyYW0gLnUzMiBwX2NhcGFjaXR5KQp7CiAgICAucmVnIC5wcmVkICVwPDQ+OwogICAgLnJlZyAuYjMyICVyPDE4PjsKICAgIC5yZWcgLmI2NCAlcmQ8MTY+OwogICAgLnJlZyAuZjMyICVmPDU+OwoKICAgIGxkLnBhcmFtLnU2NCAlcmQxLCBbcF9wcm9iXTsKICAgIGxkLnBhcmFtLnU2NCAlcmQyLCBbcF92XTsKICAgIGxkLnBhcmFtLnU2NCAlcmQzLCBbcF9vdXRdOwogICAgbGQucGFyYW0udTMyICVyMSwgW3BfbnRva107CiAgICBsZC5wYXJhbS51MzIgJXIyLCBbcF9jYXBhY2l0eV07CiAgICBjdnRhLnRvLmdsb2JhbC51NjQgJXJkNCwgJXJkMTsKICAgIGN2dGEudG8uZ2xvYmFsLnU2NCAlcmQ1LCAlcmQyOwogICAgY3Z0YS50by5nbG9iYWwudTY0ICVyZDYsICVyZDM7CgogICAgbW92LnUzMiAlcjMsICVjdGFpZC54OwogICAgc2hsLmIzMiAlcjMsICVyMywgNDsKICAgIG1vdi51MzIgJXI0LCAlY3RhaWQueTsKICAgIG1vdi51MzIgJXI1LCAldGlkLng7CiAgICBtb3YudTMyICVyNiwgJXI1OwoKVzY0X1NDQUxBUl9PVVRQVVQ6CiAgICBzZXRwLmdlLnUzMiAlcDEsICVyNiwgMTAyNDsKICAgIEAlcDEgYnJhIFc2NF9TQ0FMQVJfRE9ORTsKICAgIHNoci51MzIgJXI3LCAlcjYsIDY7CiAgICBhbmQuYjMyICVyOCwgJXI2LCA2MzsKICAgIGFkZC51MzIgJXI5LCAlcjMsICVyNzsKICAgIHNldHAuZ2UudTMyICVwMiwgJXI5LCAlcjE7CiAgICBAJXAyIGJyYSBXNjRfU0NBTEFSX05FWFQ7CgogICAgbXVsLmxvLnUzMiAlcjEwLCAlcjQsICVyMTsKICAgIGFkZC51MzIgJXIxMCwgJXIxMCwgJXI5OwogICAgbXVsLmxvLnUzMiAlcjEwLCAlcjEwLCAlcjI7CiAgICBzaGwuYjMyICVyMTAsICVyMTAsIDI7CiAgICBjdnQudTY0LnUzMiAlcmQ3LCAlcjEwOwogICAgYWRkLnU2NCAlcmQ3LCAlcmQ0LCAlcmQ3OwoKICAgIG11bC5sby51MzIgJXIxMSwgJXI0LCAlcjI7CiAgICBzaGwuYjMyICVyMTEsICVyMTEsIDY7CiAgICBhZGQudTMyICVyMTEsICVyMTEsICVyODsKICAgIHNobC5iMzIgJXIxMSwgJXIxMSwgMjsKICAgIGN2dC51NjQudTMyICVyZDgsICVyMTE7CiAgICBhZGQudTY0ICVyZDgsICVyZDUsICVyZDg7CgogICAgbW92LmYzMiAlZjEsIDBmMDAwMDAwMDA7CiAgICBtb3YudTMyICVyMTIsIDA7Clc2NF9TQ0FMQVJfSzoKICAgIHNldHAuZ2UudTMyICVwMywgJXIxMiwgJXIyOwogICAgQCVwMyBicmEgVzY0X1NDQUxBUl9TVE9SRTsKICAgIGxkLmdsb2JhbC5mMzIgJWYyLCBbJXJkN107CiAgICBsZC5nbG9iYWwuZjMyICVmMywgWyVyZDhdOwogICAgZm1hLnJuLmYzMiAlZjEsICVmMiwgJWYzLCAlZjE7CiAgICBhZGQudTY0ICVyZDcsICVyZDcsIDQ7CiAgICBhZGQudTY0ICVyZDgsICVyZDgsIDI1NjsKICAgIGFkZC51MzIgJXIxMiwgJXIxMiwgMTsKICAgIGJyYSBXNjRfU0NBTEFSX0s7CgpXNjRfU0NBTEFSX1NUT1JFOgogICAgbXVsLmxvLnUzMiAlcjEzLCAlcjQsICVyMTsKICAgIGFkZC51MzIgJXIxMywgJXIxMywgJXI5OwogICAgc2hsLmIzMiAlcjEzLCAlcjEzLCA2OwogICAgYWRkLnUzMiAlcjEzLCAlcjEzLCAlcjg7CiAgICBzaGwuYjMyICVyMTMsICVyMTMsIDI7CiAgICBjdnQudTY0LnUzMiAlcmQ5LCAlcjEzOwogICAgYWRkLnU2NCAlcmQ5LCAlcmQ2LCAlcmQ5OwogICAgc3QuZ2xvYmFsLmYzMiBbJXJkOV0sICVmMTsKClc2NF9TQ0FMQVJfTkVYVDoKICAgIGFkZC51MzIgJXI2LCAlcjYsIDEyODsKICAgIGJyYSBXNjRfU0NBTEFSX09VVFBVVDsKVzY0X1NDQUxBUl9ET05FOgogICAgcmV0Owp9CgoudmlzaWJsZSAuZW50cnkgZ2xfd2F2ZTY0X2F2X21tYTRfZjMyKAogICAgLnBhcmFtIC51NjQgcF9wcm9iLAogICAgLnBhcmFtIC51NjQgcF92dCwKICAgIC5wYXJhbSAudTY0IHBfb3V0LAogICAgLnBhcmFtIC51MzIgcF9udG9rLAogICAgLnBhcmFtIC51MzIgcF9jYXBhY2l0eSkKewogICAgLnJlZyAucHJlZCAlcDw2PjsKICAgIC5yZWcgLmIxNiAlaDwxNj47CiAgICAucmVnIC5iMzIgJXI8MzI+OwogICAgLnJlZyAuYjMyICVhX2hpMCwgJWFfaGkxLCAlYV9sbzAsICVhX2xvMTsKICAgIC5yZWcgLmIzMiAlYjBfaGksICViMF9sbywgJWIxX2hpLCAlYjFfbG87CiAgICAucmVnIC5iNjQgJXJkPDIwPjsKICAgIC5yZWcgLmYzMiAlZjwyMD47CiAgICAucmVnIC5mMzIgJWM8OD47CgogICAgbGQucGFyYW0udTY0ICVyZDEsIFtwX3Byb2JdOwogICAgbGQucGFyYW0udTY0ICVyZDIsIFtwX3Z0XTsKICAgIGxkLnBhcmFtLnU2NCAlcmQzLCBbcF9vdXRdOwogICAgbGQucGFyYW0udTMyICVyMSwgW3BfbnRva107CiAgICBsZC5wYXJhbS51MzIgJXIyLCBbcF9jYXBhY2l0eV07CiAgICBjdnRhLnRvLmdsb2JhbC51NjQgJXJkNCwgJXJkMTsKICAgIGN2dGEudG8uZ2xvYmFsLnU2NCAlcmQ1LCAlcmQyOwogICAgY3Z0YS50by5nbG9iYWwudTY0ICVyZDYsICVyZDM7CgogICAgbW92LnUzMiAlcjMsICV0aWQueDsKICAgIHNoci51MzIgJXI0LCAlcjMsIDU7CiAgICBhbmQuYjMyICVyNSwgJXIzLCAzMTsKICAgIHNoci51MzIgJXI2LCAlcjUsIDI7CiAgICBhbmQuYjMyICVyNywgJXI1LCAzOwogICAgbW92LnUzMiAlcjgsICVjdGFpZC54OwogICAgc2hsLmIzMiAlcjgsICVyOCwgNDsKICAgIG1vdi51MzIgJXI5LCAlY3RhaWQueTsKICAgIHNobC5iMzIgJXIxMCwgJXI0LCA0OwoKICAgIG1vdi5mMzIgJWMwLCAwZjAwMDAwMDAwOwogICAgbW92LmYzMiAlYzEsIDBmMDAwMDAwMDA7CiAgICBtb3YuZjMyICVjMiwgMGYwMDAwMDAwMDsKICAgIG1vdi5mMzIgJWMzLCAwZjAwMDAwMDAwOwogICAgbW92LmYzMiAlYzQsIDBmMDAwMDAwMDA7CiAgICBtb3YuZjMyICVjNSwgMGYwMDAwMDAwMDsKICAgIG1vdi5mMzIgJWM2LCAwZjAwMDAwMDAwOwogICAgbW92LmYzMiAlYzcsIDBmMDAwMDAwMDA7CiAgICBtb3YudTMyICVyMTEsIDA7CgpXNjRfTU1BX0s6CiAgICBzZXRwLmdlLnUzMiAlcDEsICVyMTEsICVyMjsKICAgIEAlcDEgYnJhIFc2NF9NTUFfU1RPUkU7CiAgICBzaGwuYjMyICVyMTIsICVyNywgMTsKICAgIGFkZC51MzIgJXIxMiwgJXIxMiwgJXIxMTsKCiAgICAvLyBBIGZyYWdtZW50OiBwcm9iYWJpbGl0eSByb3dzIGdyb3VwSUQgYW5kIGdyb3VwSUQrOCwgdHdvIEsgdmFsdWVzLgogICAgYWRkLnUzMiAlcjEzLCAlcjgsICVyNjsKICAgIG11bC5sby51MzIgJXIxNCwgJXI5LCAlcjE7CiAgICBhZGQudTMyICVyMTQsICVyMTQsICVyMTM7CiAgICBtdWwubG8udTMyICVyMTQsICVyMTQsICVyMjsKICAgIGFkZC51MzIgJXIxNCwgJXIxNCwgJXIxMjsKICAgIHNobC5iMzIgJXIxNCwgJXIxNCwgMjsKICAgIGN2dC51NjQudTMyICVyZDcsICVyMTQ7CiAgICBhZGQudTY0ICVyZDcsICVyZDQsICVyZDc7CiAgICBtb3YuZjMyICVmMSwgMGYwMDAwMDAwMDsKICAgIG1vdi5mMzIgJWYyLCAwZjAwMDAwMDAwOwogICAgc2V0cC5sdC51MzIgJXAyLCAlcjEyLCAlcjI7CiAgICBzZXRwLmx0LnUzMiAlcDMsICVyMTMsICVyMTsKICAgIGFuZC5wcmVkICVwNCwgJXAyLCAlcDM7CiAgICBAJXA0IGxkLmdsb2JhbC5mMzIgJWYxLCBbJXJkN107CiAgICBhZGQudTMyICVyMTUsICVyMTMsIDg7CiAgICBzZXRwLmx0LnUzMiAlcDMsICVyMTUsICVyMTsKICAgIGFuZC5wcmVkICVwNCwgJXAyLCAlcDM7CiAgICBAISVwNCBicmEgVzY0X0FfUk9XOF9aRVJPOwogICAgbXVsLmxvLnUzMiAlcjE2LCAlcjksICVyMTsKICAgIGFkZC51MzIgJXIxNiwgJXIxNiwgJXIxNTsKICAgIG11bC5sby51MzIgJXIxNiwgJXIxNiwgJXIyOwogICAgYWRkLnUzMiAlcjE2LCAlcjE2LCAlcjEyOwogICAgc2hsLmIzMiAlcjE2LCAlcjE2LCAyOwogICAgY3Z0LnU2NC51MzIgJXJkOCwgJXIxNjsKICAgIGFkZC51NjQgJXJkOCwgJXJkNCwgJXJkODsKICAgIGxkLmdsb2JhbC5mMzIgJWYyLCBbJXJkOF07Clc2NF9BX1JPVzhfWkVSTzoKICAgIGN2dC5ybi5mMTYuZjMyICVoMCwgJWYxOwogICAgY3Z0LnJuLmYxNi5mMzIgJWgxLCAlZjI7CiAgICBjdnQuZjMyLmYxNiAlZjMsICVoMDsKICAgIGN2dC5mMzIuZjE2ICVmNCwgJWgxOwogICAgc3ViLnJuLmYzMiAlZjUsICVmMSwgJWYzOwogICAgc3ViLnJuLmYzMiAlZjYsICVmMiwgJWY0OwogICAgY3Z0LnJuLmYxNi5mMzIgJWgyLCAlZjU7CiAgICBjdnQucm4uZjE2LmYzMiAlaDMsICVmNjsKICAgIC8vIEVhY2ggcmVnaXN0ZXIgcGFja3MgdHdvIGFkamFjZW50IEsgdmFsdWVzIGZvciBvbmUgcm93IGZyYWdtZW50LiBUaGUKICAgIC8vIHNlY29uZCB2YWx1ZSBpcyBsb2FkZWQgZXhwbGljaXRseSB0byBwcmVzZXJ2ZSB0aGUgTU1BIGxhbmUgY29udHJhY3QuCiAgICBhZGQudTMyICVyMTcsICVyMTIsIDE7CiAgICBtb3YuZjMyICVmNywgMGYwMDAwMDAwMDsKICAgIG1vdi5mMzIgJWY4LCAwZjAwMDAwMDAwOwogICAgc2V0cC5sdC51MzIgJXAyLCAlcjE3LCAlcjI7CiAgICBzZXRwLmx0LnUzMiAlcDMsICVyMTMsICVyMTsKICAgIGFuZC5wcmVkICVwNCwgJXAyLCAlcDM7CiAgICBAJXA0IGxkLmdsb2JhbC5mMzIgJWY3LCBbJXJkNys0XTsKICAgIHNldHAubHQudTMyICVwMywgJXIxNSwgJXIxOwogICAgYW5kLnByZWQgJXA0LCAlcDIsICVwMzsKICAgIEAlcDQgbGQuZ2xvYmFsLmYzMiAlZjgsIFslcmQ4KzRdOwogICAgY3Z0LnJuLmYxNi5mMzIgJWg0LCAlZjc7CiAgICBjdnQucm4uZjE2LmYzMiAlaDUsICVmODsKICAgIGN2dC5mMzIuZjE2ICVmOSwgJWg0OwogICAgY3Z0LmYzMi5mMTYgJWYxMCwgJWg1OwogICAgc3ViLnJuLmYzMiAlZjExLCAlZjcsICVmOTsKICAgIHN1Yi5ybi5mMzIgJWYxMiwgJWY4LCAlZjEwOwogICAgY3Z0LnJuLmYxNi5mMzIgJWg2LCAlZjExOwogICAgY3Z0LnJuLmYxNi5mMzIgJWg3LCAlZjEyOwogICAgbW92LmIzMiAlYV9oaTAsIHslaDAsICVoNH07CiAgICBtb3YuYjMyICVhX2hpMSwgeyVoMSwgJWg1fTsKICAgIG1vdi5iMzIgJWFfbG8wLCB7JWgyLCAlaDZ9OwogICAgbW92LmIzMiAlYV9sbzEsIHslaDMsICVoN307CgogICAgLy8gQiBmcmFnbWVudHM6IHR3byBhZGphY2VudCBOOCB0aWxlcyBvd25lZCBieSB0aGlzIHdhcnAuIFYgaXMgc3RvcmVkCiAgICAvLyBkaW1lbnNpb24tbWFqb3IsIHNvIGVhY2ggbG9naWNhbCBLeE4gb3BlcmFuZCBpcyBjb2x1bW4tbWFqb3IuCiAgICBhZGQudTMyICVyMTgsICVyMTAsICVyNjsKICAgIG11bC5sby51MzIgJXIxOSwgJXI5LCA2NDsKICAgIGFkZC51MzIgJXIxOSwgJXIxOSwgJXIxODsKICAgIG11bC5sby51MzIgJXIxOSwgJXIxOSwgJXIyOwogICAgYWRkLnUzMiAlcjE5LCAlcjE5LCAlcjEyOwogICAgc2hsLmIzMiAlcjE5LCAlcjE5LCAyOwogICAgY3Z0LnU2NC51MzIgJXJkOSwgJXIxOTsKICAgIGFkZC51NjQgJXJkOSwgJXJkNSwgJXJkOTsKICAgIGxkLmdsb2JhbC5mMzIgJWYxMywgWyVyZDldOwogICAgbGQuZ2xvYmFsLmYzMiAlZjE0LCBbJXJkOSs0XTsKICAgIGN2dC5ybi5mMTYuZjMyICVoOCwgJWYxMzsKICAgIGN2dC5ybi5mMTYuZjMyICVoOSwgJWYxNDsKICAgIGN2dC5mMzIuZjE2ICVmMTUsICVoODsKICAgIGN2dC5mMzIuZjE2ICVmMTYsICVoOTsKICAgIHN1Yi5ybi5mMzIgJWYxNywgJWYxMywgJWYxNTsKICAgIHN1Yi5ybi5mMzIgJWYxOCwgJWYxNCwgJWYxNjsKICAgIGN2dC5ybi5mMTYuZjMyICVoMTAsICVmMTc7CiAgICBjdnQucm4uZjE2LmYzMiAlaDExLCAlZjE4OwogICAgbW92LmIzMiAlYjBfaGksIHslaDgsICVoOX07CiAgICBtb3YuYjMyICViMF9sbywgeyVoMTAsICVoMTF9OwoKICAgIGFkZC51MzIgJXIyMCwgJXIxOCwgODsKICAgIG11bC5sby51MzIgJXIyMSwgJXI5LCA2NDsKICAgIGFkZC51MzIgJXIyMSwgJXIyMSwgJXIyMDsKICAgIG11bC5sby51MzIgJXIyMSwgJXIyMSwgJXIyOwogICAgYWRkLnUzMiAlcjIxLCAlcjIxLCAlcjEyOwogICAgc2hsLmIzMiAlcjIxLCAlcjIxLCAyOwogICAgY3Z0LnU2NC51MzIgJXJkMTAsICVyMjE7CiAgICBhZGQudTY0ICVyZDEwLCAlcmQ1LCAlcmQxMDsKICAgIGxkLmdsb2JhbC5mMzIgJWYxMywgWyVyZDEwXTsKICAgIGxkLmdsb2JhbC5mMzIgJWYxNCwgWyVyZDEwKzRdOwogICAgY3Z0LnJuLmYxNi5mMzIgJWgxMiwgJWYxMzsKICAgIGN2dC5ybi5mMTYuZjMyICVoMTMsICVmMTQ7CiAgICBjdnQuZjMyLmYxNiAlZjE1LCAlaDEyOwogICAgY3Z0LmYzMi5mMTYgJWYxNiwgJWgxMzsKICAgIHN1Yi5ybi5mMzIgJWYxNywgJWYxMywgJWYxNTsKICAgIHN1Yi5ybi5mMzIgJWYxOCwgJWYxNCwgJWYxNjsKICAgIGN2dC5ybi5mMTYuZjMyICVoMTQsICVmMTc7CiAgICBjdnQucm4uZjE2LmYzMiAlaDE1LCAlZjE4OwogICAgbW92LmIzMiAlYjFfaGksIHslaDEyLCAlaDEzfTsKICAgIG1vdi5iMzIgJWIxX2xvLCB7JWgxNCwgJWgxNX07CgogICAgbW1hLnN5bmMuYWxpZ25lZC5tMTZuOGs4LnJvdy5jb2wuZjMyLmYxNi5mMTYuZjMyCiAgICAgICAgeyVjMCwlYzEsJWMyLCVjM30sIHslYV9oaTAsJWFfaGkxfSwgeyViMF9oaX0sIHslYzAsJWMxLCVjMiwlYzN9OwogICAgbW1hLnN5bmMuYWxpZ25lZC5tMTZuOGs4LnJvdy5jb2wuZjMyLmYxNi5mMTYuZjMyCiAgICAgICAgeyVjMCwlYzEsJWMyLCVjM30sIHslYV9oaTAsJWFfaGkxfSwgeyViMF9sb30sIHslYzAsJWMxLCVjMiwlYzN9OwogICAgbW1hLnN5bmMuYWxpZ25lZC5tMTZuOGs4LnJvdy5jb2wuZjMyLmYxNi5mMTYuZjMyCiAgICAgICAgeyVjMCwlYzEsJWMyLCVjM30sIHslYV9sbzAsJWFfbG8xfSwgeyViMF9oaX0sIHslYzAsJWMxLCVjMiwlYzN9OwogICAgbW1hLnN5bmMuYWxpZ25lZC5tMTZuOGs4LnJvdy5jb2wuZjMyLmYxNi5mMTYuZjMyCiAgICAgICAgeyVjMCwlYzEsJWMyLCVjM30sIHslYV9sbzAsJWFfbG8xfSwgeyViMF9sb30sIHslYzAsJWMxLCVjMiwlYzN9OwogICAgbW1hLnN5bmMuYWxpZ25lZC5tMTZuOGs4LnJvdy5jb2wuZjMyLmYxNi5mMTYuZjMyCiAgICAgICAgeyVjNCwlYzUsJWM2LCVjN30sIHslYV9oaTAsJWFfaGkxfSwgeyViMV9oaX0sIHslYzQsJWM1LCVjNiwlYzd9OwogICAgbW1hLnN5bmMuYWxpZ25lZC5tMTZuOGs4LnJvdy5jb2wuZjMyLmYxNi5mMTYuZjMyCiAgICAgICAgeyVjNCwlYzUsJWM2LCVjN30sIHslYV9oaTAsJWFfaGkxfSwgeyViMV9sb30sIHslYzQsJWM1LCVjNiwlYzd9OwogICAgbW1hLnN5bmMuYWxpZ25lZC5tMTZuOGs4LnJvdy5jb2wuZjMyLmYxNi5mMTYuZjMyCiAgICAgICAgeyVjNCwlYzUsJWM2LCVjN30sIHslYV9sbzAsJWFfbG8xfSwgeyViMV9oaX0sIHslYzQsJWM1LCVjNiwlYzd9OwogICAgbW1hLnN5bmMuYWxpZ25lZC5tMTZuOGs4LnJvdy5jb2wuZjMyLmYxNi5mMTYuZjMyCiAgICAgICAgeyVjNCwlYzUsJWM2LCVjN30sIHslYV9sbzAsJWFfbG8xfSwgeyViMV9sb30sIHslYzQsJWM1LCVjNiwlYzd9OwogICAgYWRkLnUzMiAlcjExLCAlcjExLCA4OwogICAgYnJhIFc2NF9NTUFfSzsKClc2NF9NTUFfU1RPUkU6CiAgICBhZGQudTMyICVyMjIsICVyOCwgJXI2OwogICAgc2V0cC5nZS51MzIgJXA1LCAlcjIyLCAlcjE7CiAgICBAJXA1IGJyYSBXNjRfTU1BX0RPTkU7CiAgICBzaGwuYjMyICVyMjMsICVyNywgMTsKICAgIGFkZC51MzIgJXIyNCwgJXIxMCwgJXIyMzsKICAgIG11bC5sby51MzIgJXIyNSwgJXI5LCAlcjE7CiAgICBhZGQudTMyICVyMjUsICVyMjUsICVyMjI7CiAgICBzaGwuYjMyICVyMjUsICVyMjUsIDY7CiAgICBhZGQudTMyICVyMjUsICVyMjUsICVyMjQ7CiAgICBzaGwuYjMyICVyMjUsICVyMjUsIDI7CiAgICBjdnQudTY0LnUzMiAlcmQxMSwgJXIyNTsKICAgIGFkZC51NjQgJXJkMTEsICVyZDYsICVyZDExOwogICAgc3QuZ2xvYmFsLmYzMiBbJXJkMTFdLCAlYzA7CiAgICBzdC5nbG9iYWwuZjMyIFslcmQxMSs0XSwgJWMxOwogICAgc3QuZ2xvYmFsLmYzMiBbJXJkMTErMzJdLCAlYzQ7CiAgICBzdC5nbG9iYWwuZjMyIFslcmQxMSszNl0sICVjNTsKICAgIGFkZC51MzIgJXIyMiwgJXIyMiwgODsKICAgIHNldHAuZ2UudTMyICVwNSwgJXIyMiwgJXIxOwogICAgQCVwNSBicmEgVzY0X01NQV9ET05FOwogICAgYWRkLnU2NCAlcmQxMSwgJXJkMTEsIDIwNDg7CiAgICBzdC5nbG9iYWwuZjMyIFslcmQxMV0sICVjMjsKICAgIHN0Lmdsb2JhbC5mMzIgWyVyZDExKzRdLCAlYzM7CiAgICBzdC5nbG9iYWwuZjMyIFslcmQxMSszMl0sICVjNjsKICAgIHN0Lmdsb2JhbC5mMzIgWyVyZDExKzM2XSwgJWM3OwpXNjRfTU1BX0RPTkU6CiAgICByZXQ7Cn0K"},{"path":"glcuda/examples/wave64_mma_av.rs","sha256":"21fc227e11f79425dff665d280747d475c90f640f6ec7080ead646b55a3cf95b","base64":"Ly8hIFdhdmUgNjQgY29tcGVuc2F0ZWQtTU1BIEFWIGZlYXNpYmlsaXR5IHNjcmVlbi4KLy8hCi8vISBUaGlzIGxvYWRzIGFuIGlzb2xhdGVkIFBUWCBtb2R1bGUuIEl0IGRvZXMgbm90IGFsdGVyIEtlcm5lbFNldCBvciB0aGUKLy8hIHByb2R1Y3Rpb24gYXR0ZW50aW9uIGRpc3BhdGNoZXIuCgp1c2Ugc3RkOjpmZmk6OmNfdm9pZDsKdXNlIHN0ZDo6dGltZTo6SW5zdGFudDsKCnVzZSBnbGN1ZGE6OmJ1ZmZlcjo6QmFja2VuZEJ1ZmZlcjsKdXNlIGdsY3VkYTo6ZHJpdmVyOjp7Y3VkYV9hdmFpbGFibGUsIEN1ZGEsIEtlcm5lbH07CnVzZSBnbGN1ZGE6OmZmaTo6Q1VkZXZpY2VwdHI7Cgpjb25zdCBIRUFEUzogdXNpemUgPSAxNDsKY29uc3QgV0lEVEg6IHVzaXplID0gNjQ7CmNvbnN0IFdBUk1VUDogdXNpemUgPSAxMDsKY29uc3QgSVRFUlM6IHVzaXplID0gMTAwOwpjb25zdCBSRVBFQVRTOiB1c2l6ZSA9IDU7CmNvbnN0IE1BWF9BQlNfR0FURTogZjMyID0gMS4wZS01Owpjb25zdCBNSU5fUFJPRFVDVElPTl9TUEVFRFVQOiBmNjQgPSAxLjUwOwpjb25zdCBQVFg6ICZzdHIgPSBpbmNsdWRlX3N0ciEoIi4uL3NyYy9rZXJuZWxzL2dsY3VkYV9zbTc1X3dhdmU2NC5wdHgiKTsKCmZuIHZhbHVlcyhuOiB1c2l6ZSwgc2VlZDogdTY0KSAtPiBWZWM8ZjMyPiB7CiAgICBsZXQgbXV0IHN0YXRlID0gc2VlZCB8IDE7CiAgICAoMC4ubikKICAgICAgICAubWFwKHxffCB7CiAgICAgICAgICAgIHN0YXRlIF49IHN0YXRlID4+IDEyOwogICAgICAgICAgICBzdGF0ZSBePSBzdGF0ZSA8PCAyNTsKICAgICAgICAgICAgc3RhdGUgXj0gc3RhdGUgPj4gMjc7CiAgICAgICAgICAgIGxldCB1bml0ID0KICAgICAgICAgICAgICAgIChzdGF0ZS53cmFwcGluZ19tdWwoMHgyNTQ1X0Y0OTFfNEY2Q19ERDFEKSA+PiA0MCkgYXMgZjMyIC8gKDF1NjQgPDwgMjQpIGFzIGYzMjsKICAgICAgICAgICAgdW5pdCAtIDAuNQogICAgICAgIH0pCiAgICAgICAgLmNvbGxlY3QoKQp9CgpmbiBjYXVzYWxfcHJvYmFiaWxpdGllcyhudG9rOiB1c2l6ZSwgY2FwYWNpdHk6IHVzaXplKSAtPiBWZWM8ZjMyPiB7CiAgICBsZXQgcmF3ID0gdmFsdWVzKEhFQURTICogbnRvayAqIGNhcGFjaXR5LCA2NCArIGNhcGFjaXR5IGFzIHU2NCk7CiAgICBsZXQgbXV0IG91dCA9IHZlYyFbMC4wOyByYXcubGVuKCldOwogICAgZm9yIGhlYWQgaW4gMC4uSEVBRFMgewogICAgICAgIGZvciByb3cgaW4gMC4ubnRvayB7CiAgICAgICAgICAgIGxldCB2aXNpYmxlID0gKHJvdyArIDEpLm1pbihjYXBhY2l0eSk7CiAgICAgICAgICAgIGxldCBiYXNlID0gKGhlYWQgKiBudG9rICsgcm93KSAqIGNhcGFjaXR5OwogICAgICAgICAgICBsZXQgbWF4ID0gcmF3W2Jhc2UuLmJhc2UgKyB2aXNpYmxlXQogICAgICAgICAgICAgICAgLml0ZXIoKQogICAgICAgICAgICAgICAgLmNvcGllZCgpCiAgICAgICAgICAgICAgICAuZm9sZChmMzI6Ok5FR19JTkZJTklUWSwgZjMyOjptYXgpOwogICAgICAgICAgICBsZXQgbXV0IHN1bSA9IDAuMGYzMjsKICAgICAgICAgICAgZm9yIGtleSBpbiAwLi52aXNpYmxlIHsKICAgICAgICAgICAgICAgIGxldCB3ZWlnaHQgPSAocmF3W2Jhc2UgKyBrZXldIC0gbWF4KS5leHAoKTsKICAgICAgICAgICAgICAgIG91dFtiYXNlICsga2V5XSA9IHdlaWdodDsKICAgICAgICAgICAgICAgIHN1bSArPSB3ZWlnaHQ7CiAgICAgICAgICAgIH0KICAgICAgICAgICAgZm9yIGtleSBpbiAwLi52aXNpYmxlIHsKICAgICAgICAgICAgICAgIG91dFtiYXNlICsga2V5XSAvPSBzdW07CiAgICAgICAgICAgIH0KICAgICAgICB9CiAgICB9CiAgICBvdXQKfQoKZm4gdHJhbnNwb3NlX3YodjogJltmMzJdLCBjYXBhY2l0eTogdXNpemUpIC0+IFZlYzxmMzI+IHsKICAgIGxldCBtdXQgdnQgPSB2ZWMhWzAuMDsgdi5sZW4oKV07CiAgICBmb3IgaGVhZCBpbiAwLi5IRUFEUyB7CiAgICAgICAgZm9yIGtleSBpbiAwLi5jYXBhY2l0eSB7CiAgICAgICAgICAgIGZvciBkaW0gaW4gMC4uV0lEVEggewogICAgICAgICAgICAgICAgdnRbKGhlYWQgKiBXSURUSCArIGRpbSkgKiBjYXBhY2l0eSArIGtleV0gPQogICAgICAgICAgICAgICAgICAgIHZbKGhlYWQgKiBjYXBhY2l0eSArIGtleSkgKiBXSURUSCArIGRpbV07CiAgICAgICAgICAgIH0KICAgICAgICB9CiAgICB9CiAgICB2dAp9CgpmbiBsYXVuY2hfYXYoCiAgICBjdWRhOiAmQ3VkYSwKICAgIGtlcm5lbDogS2VybmVsLAogICAgcHJvYjogQ1VkZXZpY2VwdHIsCiAgICB2OiBDVWRldmljZXB0ciwKICAgIG91dDogQ1VkZXZpY2VwdHIsCiAgICBudG9rOiB1MzIsCiAgICBjYXBhY2l0eTogdTMyLAopIC0+IFJlc3VsdDwoKSwgZ2xjb3JlOjpHbEVycm9yPiB7CiAgICBsZXQgKG11dCBwcm9iLCBtdXQgdiwgbXV0IG91dCwgbXV0IG50b2ssIG11dCBjYXBhY2l0eSkgPSAocHJvYiwgdiwgb3V0LCBudG9rLCBjYXBhY2l0eSk7CiAgICBsZXQgbXV0IHBhcmFtcyA9IFsKICAgICAgICAmbXV0IHByb2IgYXMgKm11dCBfIGFzICptdXQgY192b2lkLAogICAgICAgICZtdXQgdiBhcyAqbXV0IF8gYXMgKm11dCBjX3ZvaWQsCiAgICAgICAgJm11dCBvdXQgYXMgKm11dCBfIGFzICptdXQgY192b2lkLAogICAgICAgICZtdXQgbnRvayBhcyAqbXV0IF8gYXMgKm11dCBjX3ZvaWQsCiAgICAgICAgJm11dCBjYXBhY2l0eSBhcyAqbXV0IF8gYXMgKm11dCBjX3ZvaWQsCiAgICBdOwogICAgY3VkYS5sYXVuY2goCiAgICAgICAga2VybmVsLAogICAgICAgIChudG9rLmRpdl9jZWlsKDE2KSwgSEVBRFMgYXMgdTMyLCAxKSwKICAgICAgICAoMTI4LCAxLCAxKSwKICAgICAgICAwLAogICAgICAgICZtdXQgcGFyYW1zLAogICAgKQp9CgpmbiB0aW1lZDxGPihjdWRhOiAmQ3VkYSwgbXV0IGxhdW5jaDogRikgLT4gUmVzdWx0PGY2NCwgZ2xjb3JlOjpHbEVycm9yPgp3aGVyZQogICAgRjogRm5NdXQoKSAtPiBSZXN1bHQ8KCksIGdsY29yZTo6R2xFcnJvcj4sCnsKICAgIGZvciBfIGluIDAuLldBUk1VUCB7CiAgICAgICAgbGF1bmNoKCk/OwogICAgfQogICAgY3VkYS5zeW5jaHJvbml6ZSgpPzsKICAgIGxldCBzdGFydCA9IEluc3RhbnQ6Om5vdygpOwogICAgZm9yIF8gaW4gMC4uSVRFUlMgewogICAgICAgIGxhdW5jaCgpPzsKICAgIH0KICAgIGN1ZGEuc3luY2hyb25pemUoKT87CiAgICBPayhzdGFydC5lbGFwc2VkKCkuYXNfc2Vjc19mNjQoKSAqIDEuMGU2IC8gSVRFUlMgYXMgZjY0KQp9CgpmbiBtZWRpYW4oc2FtcGxlczogJm11dCBbZjY0XSkgLT4gZjY0IHsKICAgIHNhbXBsZXMuc29ydF9ieShmNjQ6OnRvdGFsX2NtcCk7CiAgICAoc2FtcGxlc1tzYW1wbGVzLmxlbigpIC8gMiAtIDFdICsgc2FtcGxlc1tzYW1wbGVzLmxlbigpIC8gMl0pICogMC41Cn0KCnN0cnVjdCBSZWNvcmQgewogICAgY2FwYWNpdHk6IHVzaXplLAogICAgc2NhbGFyX3VzOiBmNjQsCiAgICBtbWFfdXM6IGY2NCwKICAgIG1heF9hYnM6IGYzMiwKICAgIHJtczogZjY0LAp9CgppbXBsIFJlY29yZCB7CiAgICBmbiBqc29uKCZzZWxmKSAtPiBTdHJpbmcgewogICAgICAgIGZvcm1hdCEoCiAgICAgICAgICAgICJ7e1wiY2FwYWNpdHlcIjp7fSxcInNjYWxhcl91c1wiOns6LjN9LFwibW1hX3VzXCI6ezouM30sXAogICAgICAgICAgICAgXCJzcGVlZHVwXCI6ezouNH0sXCJtYXhfYWJzXCI6ezouOWV9LFwicm1zXCI6ezouOWV9fX0iLAogICAgICAgICAgICBzZWxmLmNhcGFjaXR5LAogICAgICAgICAgICBzZWxmLnNjYWxhcl91cywKICAgICAgICAgICAgc2VsZi5tbWFfdXMsCiAgICAgICAgICAgIHNlbGYuc2NhbGFyX3VzIC8gc2VsZi5tbWFfdXMsCiAgICAgICAgICAgIHNlbGYubWF4X2FicywKICAgICAgICAgICAgc2VsZi5ybXMsCiAgICAgICAgKQogICAgfQp9CgpmbiBzY3JlZW4oCiAgICBjdWRhOiAmQ3VkYSwKICAgIHNjYWxhcjogS2VybmVsLAogICAgbW1hOiBLZXJuZWwsCiAgICBjYXBhY2l0eTogdXNpemUsCikgLT4gUmVzdWx0PFJlY29yZCwgQm94PGR5biBzdGQ6OmVycm9yOjpFcnJvcj4+IHsKICAgIGxldCBudG9rID0gY2FwYWNpdHk7CiAgICBsZXQgcHJvYiA9IGNhdXNhbF9wcm9iYWJpbGl0aWVzKG50b2ssIGNhcGFjaXR5KTsKICAgIGxldCB2ID0gdmFsdWVzKEhFQURTICogY2FwYWNpdHkgKiBXSURUSCwgNjQwMCArIGNhcGFjaXR5IGFzIHU2NCk7CiAgICBsZXQgdnQgPSB0cmFuc3Bvc2VfdigmdiwgY2FwYWNpdHkpOwogICAgbGV0IG91dHB1dF9sZW4gPSBIRUFEUyAqIG50b2sgKiBXSURUSDsKICAgIGxldCBieXRlcyA9ICgocHJvYi5sZW4oKSArIHYubGVuKCkgKyB2dC5sZW4oKSArIDIgKiBvdXRwdXRfbGVuKSAqIDQgKyAxXzA0OF81NzYpIGFzIHU2NDsKICAgIGxldCBtdXQgYnVmZmVyID0gQmFja2VuZEJ1ZmZlcjo6bmV3KGN1ZGEsIGJ5dGVzKT87CiAgICBsZXQgZHByb2IgPSBidWZmZXIuYWxsb2NfZjMyKHByb2IubGVuKCkpPy5kcHRyOwogICAgbGV0IGR2ID0gYnVmZmVyLmFsbG9jX2YzMih2LmxlbigpKT8uZHB0cjsKICAgIGxldCBkdnQgPSBidWZmZXIuYWxsb2NfZjMyKHZ0LmxlbigpKT8uZHB0cjsKICAgIGxldCBkc2NhbGFyID0gYnVmZmVyLmFsbG9jX2YzMihvdXRwdXRfbGVuKT8uZHB0cjsKICAgIGxldCBkbW1hID0gYnVmZmVyLmFsbG9jX2YzMihvdXRwdXRfbGVuKT8uZHB0cjsKICAgIGN1ZGEuaHRvZF9mMzIoZHByb2IsICZwcm9iKT87CiAgICBjdWRhLmh0b2RfZjMyKGR2LCAmdik/OwogICAgY3VkYS5odG9kX2YzMihkdnQsICZ2dCk/OwoKICAgIGxldCBzY2FsYXJfbGF1bmNoID0gfHwgewogICAgICAgIGxhdW5jaF9hdigKICAgICAgICAgICAgY3VkYSwKICAgICAgICAgICAgc2NhbGFyLAogICAgICAgICAgICBkcHJvYiwKICAgICAgICAgICAgZHYsCiAgICAgICAgICAgIGRzY2FsYXIsCiAgICAgICAgICAgIG50b2sgYXMgdTMyLAogICAgICAgICAgICBjYXBhY2l0eSBhcyB1MzIsCiAgICAgICAgKQogICAgfTsKICAgIGxldCBtbWFfbGF1bmNoID0gfHwgbGF1bmNoX2F2KGN1ZGEsIG1tYSwgZHByb2IsIGR2dCwgZG1tYSwgbnRvayBhcyB1MzIsIGNhcGFjaXR5IGFzIHUzMik7CiAgICBzY2FsYXJfbGF1bmNoKCk/OwogICAgbW1hX2xhdW5jaCgpPzsKICAgIGN1ZGEuc3luY2hyb25pemUoKT87CiAgICBsZXQgbXV0IHNjYWxhcl9ob3N0ID0gdmVjIVswLjA7IG91dHB1dF9sZW5dOwogICAgbGV0IG11dCBtbWFfaG9zdCA9IHZlYyFbMC4wOyBvdXRwdXRfbGVuXTsKICAgIGN1ZGEuZHRvaF9mMzIoJm11dCBzY2FsYXJfaG9zdCwgZHNjYWxhcik/OwogICAgY3VkYS5kdG9oX2YzMigmbXV0IG1tYV9ob3N0LCBkbW1hKT87CiAgICBpZiAhc2NhbGFyX2hvc3QKICAgICAgICAuaXRlcigpCiAgICAgICAgLmNoYWluKCZtbWFfaG9zdCkKICAgICAgICAuYWxsKHx2YWx1ZXwgdmFsdWUuaXNfZmluaXRlKCkpCiAgICB7CiAgICAgICAgcmV0dXJuIEVycihmb3JtYXQhKCJub24tZmluaXRlIFdhdmUgNjQgb3V0cHV0IGF0IGNhcGFjaXR5IHtjYXBhY2l0eX0iKS5pbnRvKCkpOwogICAgfQogICAgbGV0IG11dCBtYXhfYWJzID0gMC4wZjMyOwogICAgbGV0IG11dCBzcXVhcmVkID0gMC4wZjY0OwogICAgZm9yICgmbGVmdCwgJnJpZ2h0KSBpbiBzY2FsYXJfaG9zdC5pdGVyKCkuemlwKCZtbWFfaG9zdCkgewogICAgICAgIGxldCBkZWx0YSA9IChsZWZ0IC0gcmlnaHQpLmFicygpOwogICAgICAgIG1heF9hYnMgPSBtYXhfYWJzLm1heChkZWx0YSk7CiAgICAgICAgc3F1YXJlZCArPSBmNjQ6OmZyb20oZGVsdGEpLnBvd2koMik7CiAgICB9CiAgICBsZXQgcm1zID0gKHNxdWFyZWQgLyBvdXRwdXRfbGVuIGFzIGY2NCkuc3FydCgpOwoKICAgIGxldCAobXV0IHNjYWxhcl9zYW1wbGVzLCBtdXQgbW1hX3NhbXBsZXMpID0gKAogICAgICAgIFZlYzo6d2l0aF9jYXBhY2l0eSgyICogUkVQRUFUUyksCiAgICAgICAgVmVjOjp3aXRoX2NhcGFjaXR5KDIgKiBSRVBFQVRTKSwKICAgICk7CiAgICBmb3IgXyBpbiAwLi5SRVBFQVRTIHsKICAgICAgICBzY2FsYXJfc2FtcGxlcy5wdXNoKHRpbWVkKGN1ZGEsIHNjYWxhcl9sYXVuY2gpPyk7CiAgICAgICAgbW1hX3NhbXBsZXMucHVzaCh0aW1lZChjdWRhLCBtbWFfbGF1bmNoKT8pOwogICAgICAgIG1tYV9zYW1wbGVzLnB1c2godGltZWQoY3VkYSwgbW1hX2xhdW5jaCk/KTsKICAgICAgICBzY2FsYXJfc2FtcGxlcy5wdXNoKHRpbWVkKGN1ZGEsIHNjYWxhcl9sYXVuY2gpPyk7CiAgICB9CiAgICBsZXQgc2NhbGFyX3VzID0gbWVkaWFuKCZtdXQgc2NhbGFyX3NhbXBsZXMpOwogICAgbGV0IG1tYV91cyA9IG1lZGlhbigmbXV0IG1tYV9zYW1wbGVzKTsKICAgIGJ1ZmZlci5mcmVlKGN1ZGEpPzsKICAgIE9rKFJlY29yZCB7CiAgICAgICAgY2FwYWNpdHksCiAgICAgICAgc2NhbGFyX3VzLAogICAgICAgIG1tYV91cywKICAgICAgICBtYXhfYWJzLAogICAgICAgIHJtcywKICAgIH0pCn0KCmZuIG1haW4oKSAtPiBSZXN1bHQ8KCksIEJveDxkeW4gc3RkOjplcnJvcjo6RXJyb3I+PiB7CiAgICBpZiAhY3VkYV9hdmFpbGFibGUoKSB7CiAgICAgICAgcHJpbnRsbiEoIlt3YXZlNjQtYXZdIG5vIENVREEgZGV2aWNlOyBub3RoaW5nIG1lYXN1cmVkIik7CiAgICAgICAgcmV0dXJuIE9rKCgpKTsKICAgIH0KICAgIGxldCBjdWRhID0gQ3VkYTo6cHJvYmUoKT87CiAgICBpZiAoY3VkYS5pbmZvLnNtX21ham9yLCBjdWRhLmluZm8uc21fbWlub3IpICE9ICg3LCA1KSB7CiAgICAgICAgcmV0dXJuIEVycihmb3JtYXQhKAogICAgICAgICAgICAiV2F2ZSA2NCByZXF1aXJlcyBzbV83NSwgZ290IHNtX3t9e30iLAogICAgICAgICAgICBjdWRhLmluZm8uc21fbWFqb3IsIGN1ZGEuaW5mby5zbV9taW5vcgogICAgICAgICkKICAgICAgICAuaW50bygpKTsKICAgIH0KICAgIGxldCBtb2R1bGUgPSBjdWRhLmxvYWRfbW9kdWxlKFBUWCk/OwogICAgbGV0IHNjYWxhciA9IG1vZHVsZS5nZXRfZnVuY3Rpb24oImdsX3dhdmU2NF9hdl9zY2FsYXJfZjMyIik/OwogICAgbGV0IG1tYSA9IG1vZHVsZS5nZXRfZnVuY3Rpb24oImdsX3dhdmU2NF9hdl9tbWE0X2YzMiIpPzsKICAgIGxldCBzY2FsYXJfYmxvY2tzID0gY3VkYQogICAgICAgIC5tYXhfYWN0aXZlX2Jsb2Nrc19wZXJfc20oc2NhbGFyLCAxMjgsIDApCiAgICAgICAgLm9rX29yKCJzY2FsYXIgb2NjdXBhbmN5IHVuYXZhaWxhYmxlIik/OwogICAgbGV0IG1tYV9ibG9ja3MgPSBjdWRhCiAgICAgICAgLm1heF9hY3RpdmVfYmxvY2tzX3Blcl9zbShtbWEsIDEyOCwgMCkKICAgICAgICAub2tfb3IoIk1NQSBvY2N1cGFuY3kgdW5hdmFpbGFibGUiKT87CiAgICBsZXQgcmVjb3JkcyA9IFsKICAgICAgICBzY3JlZW4oJmN1ZGEsIHNjYWxhciwgbW1hLCAxKT8sCiAgICAgICAgc2NyZWVuKCZjdWRhLCBzY2FsYXIsIG1tYSwgMTcpPywKICAgICAgICBzY3JlZW4oJmN1ZGEsIHNjYWxhciwgbW1hLCAyNDEpPywKICAgICAgICBzY3JlZW4oJmN1ZGEsIHNjYWxhciwgbW1hLCAyNDQpPywKICAgIF07CiAgICBwcmludGxuISgKICAgICAgICAiW3dhdmU2NC1yZXNvdXJjZV0ge3tcInNjYWxhcl9ibG9ja3NfcGVyX3NtXCI6e30sXCJtbWFfYmxvY2tzX3Blcl9zbVwiOnt9LFwKICAgICAgICAgXCJ0aHJlYWRzXCI6MTI4LFwiZHluYW1pY19zaGFyZWRfYnl0ZXNcIjowfX0iLAogICAgICAgIHNjYWxhcl9ibG9ja3MsIG1tYV9ibG9ja3MsCiAgICApOwogICAgcHJpbnRsbiEoCiAgICAgICAgIlt3YXZlNjQtYXZdIHt7XCJ3YXJtdXBcIjp7fSxcIml0ZXJzXCI6e30sXCJyZXBlYXRzXCI6e30sXCJyZWNvcmRzXCI6W3t9XX19IiwKICAgICAgICBXQVJNVVAsCiAgICAgICAgSVRFUlMsCiAgICAgICAgUkVQRUFUUywKICAgICAgICByZWNvcmRzCiAgICAgICAgICAgIC5pdGVyKCkKICAgICAgICAgICAgLm1hcChSZWNvcmQ6Ompzb24pCiAgICAgICAgICAgIC5jb2xsZWN0Ojo8VmVjPF8+PigpCiAgICAgICAgICAgIC5qb2luKCIsIiksCiAgICApOwogICAgaWYgbGV0IFNvbWUocmVjb3JkKSA9IHJlY29yZHMuaXRlcigpLmZpbmQofHJlY29yZHwgcmVjb3JkLm1heF9hYnMgPiBNQVhfQUJTX0dBVEUpIHsKICAgICAgICByZXR1cm4gRXJyKGZvcm1hdCEoCiAgICAgICAgICAgICJXYXZlIDY0IG51bWVyaWMgZ2F0ZSBmYWlsZWQgYXQgY2FwYWNpdHkge306IHs6LjllfSA+IHs6LjFlfSIsCiAgICAgICAgICAgIHJlY29yZC5jYXBhY2l0eSwgcmVjb3JkLm1heF9hYnMsIE1BWF9BQlNfR0FURQogICAgICAgICkKICAgICAgICAuaW50bygpKTsKICAgIH0KICAgIGxldCBwcm9kdWN0aW9uID0gcmVjb3Jkcy5sYXN0KCkuZXhwZWN0KCJyZWNvcmRzIGlzIG5vbi1lbXB0eSIpOwogICAgbGV0IHNwZWVkdXAgPSBwcm9kdWN0aW9uLnNjYWxhcl91cyAvIHByb2R1Y3Rpb24ubW1hX3VzOwogICAgaWYgc3BlZWR1cCA8IE1JTl9QUk9EVUNUSU9OX1NQRUVEVVAgewogICAgICAgIHJldHVybiBFcnIoZm9ybWF0ISgKICAgICAgICAgICAgIldhdmUgNjQgc3BlZWQgZ2F0ZSBmYWlsZWQ6IHtzcGVlZHVwOi40fXggPCB7TUlOX1BST0RVQ1RJT05fU1BFRURVUDouMn14IgogICAgICAgICkKICAgICAgICAuaW50bygpKTsKICAgIH0KICAgIE9rKCgpKQp9Cg=="}]')
ROOT = Path("/kaggle/working/wave66")
TREE = ROOT / "repo"
RESULTS = ROOT / "results"
ARCHIVE = Path("/kaggle/working/glcuda-t4-wave66-mma-av-results.zip")

def run(cmd, cwd=None, env=None, timeout=3600, check=True):
    merged = os.environ.copy()
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    p = subprocess.run([str(x) for x in cmd], cwd=cwd, env=merged, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout)
    print("$", " ".join(str(x) for x in cmd), flush=True)
    print(p.stdout[-12000:], flush=True)
    print(p.stderr[-12000:], file=sys.stderr, flush=True)
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p

def save(name, process):
    (RESULTS / name).write_text(
        process.stdout + "\n--- STDERR ---\n" + process.stderr,
        encoding="utf-8",
    )

def archive():
    if ARCHIVE.exists():
        ARCHIVE.unlink()
    with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(RESULTS.rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RESULTS))
    print("ARCHIVE", ARCHIVE, hashlib.sha256(ARCHIVE.read_bytes()).hexdigest())

try:
    shutil.rmtree(ROOT, ignore_errors=True)
    RESULTS.mkdir(parents=True)
    gpu = run(["nvidia-smi", "--query-gpu=name,compute_cap,driver_version",
               "--format=csv,noheader,nounits"], timeout=60)
    save("nvidia-smi.log", gpu)
    gpu_line = gpu.stdout.strip().splitlines()[0]
    if "Tesla T4" not in gpu_line or "7.5" not in gpu_line:
        raise RuntimeError(f"Wave 66 requires Tesla T4 sm_75, got {gpu_line}")

    if not shutil.which("cargo"):
        installer = ROOT / "rustup-init.sh"
        urllib.request.urlretrieve("https://sh.rustup.rs", installer)
        run(["sh", installer, "-y", "--profile", "minimal"], timeout=1800)
        os.environ["PATH"] = str(Path.home() / ".cargo/bin") + os.pathsep + os.environ["PATH"]
    cargo = shutil.which("cargo") or str(Path.home() / ".cargo/bin/cargo")
    save("rust.log", run([cargo, "--version"], timeout=60))

    save("git-clone.log", run(["git", "clone", "--filter=blob:none", REPO_URL, TREE], timeout=1800))
    save("git-checkout.log", run(["git", "checkout", "--detach", BASE_REV], cwd=TREE, timeout=300))
    manifest = []
    for item in EMBEDDED:
        data = base64.b64decode(item["base64"])
        actual = hashlib.sha256(data).hexdigest()
        if actual != item["sha256"]:
            raise RuntimeError(f"embedded SHA mismatch: {item['path']}")
        path = TREE / item["path"]
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(data)
        manifest.append({"path": item["path"], "sha256": actual, "bytes": len(data)})
    (RESULTS / "manifest.json").write_text(json.dumps(manifest, indent=2))
    save("git-diff-check.log", run(["git", "diff", "--check"], cwd=TREE))

    ptx_path = TREE / "glcuda/src/kernels/glcuda_sm75_wave64.ptx"
    ptxas = shutil.which("ptxas") or "/usr/local/cuda/bin/ptxas"
    if not Path(ptxas).is_file():
        raise RuntimeError(f"ptxas not found: {ptxas}")
    assembled = run([ptxas, "-v", "-arch=sm_75", ptx_path,
                     "-o", ROOT / "wave66.cubin"], cwd=TREE, timeout=1800)
    save("ptxas-v.log", assembled)
    ptx_log = assembled.stdout + "\n" + assembled.stderr
    resources = {}
    for entry in ["gl_wave64_av_scalar_f32", "gl_wave64_av_mma4_f32"]:
        match = re.search(r"Compiling entry function ['\"]" + re.escape(entry) +
                          r"['\"].*?(?=Compiling entry function|\Z)", ptx_log, re.S)
        segment = match.group(0) if match else ""
        reg = re.search(r"Used (\d+) registers", segment)
        resources[entry] = {
            "found": bool(match),
            "registers": int(reg.group(1)) if reg else None,
            "spill_stores": max([int(x) for x in re.findall(r"(\d+) bytes spill stores", segment)] or [0]),
            "spill_loads": max([int(x) for x in re.findall(r"(\d+) bytes spill loads", segment)] or [0]),
            "stack": max([int(x) for x in re.findall(r"(\d+) bytes stack frame", segment)] or [0]),
        }
    (RESULTS / "resources.json").write_text(json.dumps(resources, indent=2))
    if not all(x["found"] for x in resources.values()):
        raise RuntimeError(f"missing ptxas entry: {resources}")
    if any(x["spill_stores"] or x["spill_loads"] or x["stack"] for x in resources.values()):
        raise RuntimeError(f"resource gate failed: {resources}")

    build = run([cargo, "build", "--release", "-p", "glcuda", "--example",
                 "wave64_mma_av", "--locked"], cwd=TREE, timeout=7200)
    save("cargo-build.log", build)
    measured = run([TREE / "target/release/examples/wave64_mma_av"], cwd=TREE,
                   env={"GLCUDA_JIT_VERBOSE": "1", "CUDA_VISIBLE_DEVICES": "0"},
                   timeout=7200, check=False)
    save("wave66-direct.log", measured)
    result_line = next((x for x in measured.stdout.splitlines() if x.startswith("[wave64-av] ")), "")
    resource_line = next((x for x in measured.stdout.splitlines() if x.startswith("[wave64-resource] ")), "")
    result = {
        "base_revision": BASE_REV,
        "gpu": gpu_line,
        "ptxas": resources,
        "driver": json.loads(resource_line.split("] ", 1)[1]) if resource_line else None,
        "direct": json.loads(result_line.split("] ", 1)[1]) if result_line else None,
        "returncode": measured.returncode,
    }
    (RESULTS / "result.json").write_text(json.dumps(result, indent=2))
    if measured.returncode:
        raise RuntimeError("Wave 66 direct gate failed")
    (RESULTS / "PASS.json").write_text(json.dumps({"status": "feasibility_pass"}, indent=2))
    archive()
except Exception:
    RESULTS.mkdir(parents=True, exist_ok=True)
    (RESULTS / "FAILED.txt").write_text(traceback.format_exc())
    archive()
    raise
